<a href="https://colab.research.google.com/github/jorgemunozl/vla-test/blob/main/sixth_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test 16 Fast In Slow

How "Async" works.

In [1]:
import os

# Test
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
print(torch.cuda.device_count()) 

1


In [2]:
1# DS_ID = "NONHUMAN-RESEARCH/general-task-index"
DS_ID = "NONHUMAN-RESEARCH/pick-and-place-all-fruits-annotated"
PRETRAINED_PATH = "lerobot/pi05_base"

In [3]:
from xhuman.policies.fis.configuration_fis import FISConfig

policy_config = FISConfig(repo_id="none",device="cuda",pretrained_path=PRETRAINED_PATH)

In [4]:
from xhuman.configs.default import LerobotDatasetConfig

dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)

In [5]:
from xhuman.configs.train import TrainPipelineConfigXHUMAN

cfg = TrainPipelineConfigXHUMAN(
    dataset=dataset_config,
    policy=policy_config,
)
cfg.validate()

In [6]:
from xhuman.datasets.factory import make_dataset_xhuman

dataset = make_dataset_xhuman(cfg)

In [7]:
from xhuman.policies.factory import make_xhuman_policy

policy = make_xhuman_policy(
    cfg=cfg.policy,
    ds_meta=dataset.meta,
)

TypeError: SplitPaliGemmaModel.__init__() missing 1 required positional argument: 'action_expert_config'

In [ ]:
from xhuman.policies.factory import make_xhuman_pre_post_processors

preprocessor, _ = make_xhuman_pre_post_processors(
        policy_cfg=policy_config,
        dataset_stats=dataset.meta.stats,
    )

In [ ]:
device = torch.device("cuda")

In [ ]:
frame = dataset[0]
frame.keys()

dict_keys(['observation.images.left', 'observation.images.top', 'observation.images.right', 'action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'general_task_index', 'action_is_pad', 'task', 'general_task'])

In [ ]:
batch_slow = preprocessor.slow(frame)

In [ ]:
batch_fast = preprocessor.fast(frame)

In [ ]:
batch_fast["observation.state"]

tensor([[[-8.0769e+02, -1.0988e+07, -1.0000e+00,  3.1662e+00,  4.5636e-01,
          -3.5195e+00, -1.7731e+01,  9.4314e-01, -9.9952e-01,  9.9899e-01,
           1.9229e-01, -7.0983e-01,  1.0242e+00, -1.0009e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00]]], device='cuda:0')

In [ ]:
batch_slow.keys()

dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'action_is_pad', 'task', 'index', 'task_index', 'episode_index', 'observation.images.left', 'observation.images.top', 'observation.images.right', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask'])

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")

In [ ]:
tokens_pt = batch_slow["observation.language.tokens"]
words = tokenizer.batch_decode(tokens_pt, skip_special_tokens=True)
words

['pick up the grape and put it in the basket']

In [ ]:
prefix_out , kv_cache , pad_mask  = policy.sample_embedding(batch_slow)

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'shape'

In [ ]:
images, image_masks = policy._preprocess_images(batch_slow)

In [ ]:
embeds = policy.model.paligemma_with_expert.embed_image(images[0])

In [ ]:
embeds.pooler_output

tensor([[[ 1.1416e-01, -1.7856e-02,  1.1563e-02,  ...,  5.6558e-03,
           8.0627e-03,  4.4585e-02],
         [ 5.2497e-02,  1.1205e-02,  1.3555e-02,  ...,  2.4782e-05,
           1.5854e-02,  1.0554e-02],
         [ 1.2712e-01, -2.1130e-02,  1.0253e-02,  ...,  1.6241e-03,
           5.6408e-03,  4.3863e-02],
         ...,
         [ 4.1675e-02,  4.8902e-04,  4.2399e-03,  ..., -6.0740e-03,
          -3.2375e-03, -2.3513e-03],
         [ 8.6982e-02, -7.5749e-03,  2.1022e-02,  ..., -2.7076e-02,
           3.0097e-03,  1.8997e-02],
         [ 1.2004e-01, -9.9731e-03,  1.7933e-02,  ..., -3.2756e-02,
          -3.8281e-03,  8.3293e-03]]], device='cuda:0', grad_fn=<DivBackward0>)

In [ ]:
last = embeds.last_hidden_state
embeds.pooler_output.shape

torch.Size([1, 256, 2048])

In [ ]:
print(last.shape)

torch.Size([1, 256, 1152])


In [ ]:
prefix_out[0].shape

In [ ]:
batch_slow.keys()

In [ ]:
img_fast, img_masks_fast = policy._preprocess_images(batch_fast)
raw_state = batch_fast["observation.state"]
raw_state

In [ ]:
batch_slow["observation.state"]

In [ ]:
policy.model.config.output_features["action"].shape